# ラベル伝播と相互 k 近傍グラフを自分で実装して、少数のラベルを広げる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/B_implementations/label_propagation_scratch.ipynb)

NLP クックブック B-09 ／ 仕組みを理解するための実装

ラベル付きの文書がごく少ししかないとき、ラベルのない大量の文書も使って分類の精度を上げたい。

In [1]:
!pip install -q scikit-learn fugashi unidic-lite

In [2]:
import io, re, tarfile, urllib.request
import numpy as np
import pandas as pd
from fugashi import Tagger
from sklearn.feature_extraction.text import TfidfVectorizer

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] not in ("LICENSE.txt", "README.txt"):
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "text": lines[2] + "\n" + "\n".join(lines[3:])[:600]})
    return pd.DataFrame(rows)

df = load_livedoor().sample(3000, random_state=0).reset_index(drop=True)
tagger = Tagger()
tokenize = lambda t: [w.surface for w in tagger(t) if w.feature.pos1 in ("名詞", "動詞", "形容詞")]
X = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, min_df=3, max_df=0.5, sublinear_tf=True).fit_transform(df["text"])
y = pd.factorize(df["label"])[0]
print(X.shape, df["label"].nunique(), "クラス")

(3000, 12824) 9 クラス


In [3]:
import scipy.sparse as sp

S = (X @ X.T).toarray()                    # TF-IDF は正規化済みなので、内積がコサイン類似度
np.fill_diagonal(S, 0)

def knn_graph(S, k, mutual):
    """各文書から類似度の高い k 個に辺を張る。mutual=True なら、互いに k 近傍に入っている組だけを残す"""
    n = len(S)
    idx = np.argsort(-S, axis=1)[:, :k]
    A = sp.csr_matrix((np.ones(n * k), (np.repeat(np.arange(n), k), idx.ravel())), shape=(n, n))
    A = A.multiply(A.T) if mutual else A.maximum(A.T)
    return sp.csr_matrix(A.multiply(S))    # 辺の重みは類似度

for k in [10, 30]:
    for mutual in [False, True]:
        A = knn_graph(S, k, mutual)
        degree = np.asarray((A > 0).sum(axis=1)).ravel()
        same = (y[A.nonzero()[0]] == y[A.nonzero()[1]]).mean()
        print(f"k={k} {'相互 k 近傍' if mutual else 'k 近傍'}: 辺 {A.nnz // 2} / 次数の最大 {degree.max()} / 孤立した文書 {(degree == 0).sum()} / 同じクラスどうしを結ぶ辺の割合 {same:.3f}")

k=10 k 近傍: 辺 20825 / 次数の最大 101 / 孤立した文書 0 / 同じクラスどうしを結ぶ辺の割合 0.682
k=10 相互 k 近傍: 辺 9175 / 次数の最大 10 / 孤立した文書 32 / 同じクラスどうしを結ぶ辺の割合 0.760
k=30 k 近傍: 辺 61293 / 次数の最大 218 / 孤立した文書 0 / 同じクラスどうしを結ぶ辺の割合 0.608
k=30 相互 k 近傍: 辺 28707 / 次数の最大 30 / 孤立した文書 1 / 同じクラスどうしを結ぶ辺の割合 0.697


In [4]:
def label_propagation(A, y, labeled, alpha=0.9, iterations=50):
    """ラベル拡散法: 隣の文書のスコアを受け取ることを繰り返す（alpha は隣からもらう割合）"""
    d = np.asarray(A.sum(axis=1)).ravel()
    d_inv = np.where(d > 0, 1 / np.sqrt(np.maximum(d, 1e-12)), 0)
    W = sp.diags(d_inv) @ A @ sp.diags(d_inv)
    Y = np.zeros((len(y), y.max() + 1)); Y[labeled, y[labeled]] = 1
    F = Y.copy()
    for _ in range(iterations):
        F = alpha * (W @ F) + (1 - alpha) * Y
    return F.argmax(axis=1)

from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(0)
results = []
for per_class in [2, 5, 20]:
    for trial in range(5):
        labeled = np.concatenate([rng.choice(np.where(y == c)[0], per_class, replace=False) for c in range(y.max() + 1)])
        rest = np.setdiff1d(np.arange(len(y)), labeled)
        row = {"1 クラスあたりのラベル数": per_class,
               "ロジスティック回帰 (ラベル付きのみ)": (LogisticRegression(C=10, max_iter=1000).fit(X[labeled], y[labeled]).predict(X[rest]) == y[rest]).mean()}
        for name, mutual in [("ラベル伝播 (k 近傍)", False), ("ラベル伝播 (相互 k 近傍)", True)]:
            row[name] = (label_propagation(knn_graph(S, 30, mutual), y, labeled)[rest] == y[rest]).mean()
        results.append(row)
pd.DataFrame(results).groupby("1 クラスあたりのラベル数").mean().round(3)

,ロジスティック回帰 (ラベル付きのみ),ラベル伝播 (k 近傍),ラベル伝播 (相互 k 近傍)
1 クラスあたりのラベル数,,,
2,0.508,0.624,0.607
5,0.635,0.669,0.686
20,0.784,0.774,0.771


In [5]:
# k 近傍グラフでは、多くの文書の近傍に入る「ハブ」ができる。ハブの辺の多くは一方向（相手の k 近傍にハブが入っているだけ）で、
# 一方向の辺は、互いに近傍である辺より別のクラスどうしを結びやすい。相互 k 近傍グラフは一方向の辺を捨てる
A = knn_graph(S, 30, mutual=False)
M = knn_graph(S, 30, mutual=True)
rows, cols = sp.triu(A).nonzero()
kept = np.asarray(M[rows, cols]).ravel() > 0
same = y[rows] == y[cols]
degree = np.asarray((A > 0).sum(axis=1)).ravel()
print(f"同じクラスどうしを結ぶ辺の割合: 互いに近傍の辺 {same[kept].mean():.3f} / 一方向の辺 {same[~kept].mean():.3f} / 次数の中央値 {np.median(degree):.0f}")

own = np.argsort(-S, axis=1)[:, :30]           # 各文書自身の k 近傍
for h in np.argsort(-degree)[:5]:
    neighbors = A[h].nonzero()[1]
    one_way = np.setdiff1d(neighbors, own[h])   # ハブを k 近傍に入れているが、ハブの k 近傍には入らない文書
    print(f"次数 {degree[h]} / クラス {df['label'][h]} / 同じクラスの割合: 自身の k 近傍 {np.mean(y[own[h]] == y[h]):.2f}, 一方向の近傍 {len(one_way)} 件 {np.mean(y[one_way] == y[h]):.2f}"
          f" / 別のクラスの近傍 {(y[neighbors] != y[h]).sum()} 件 | {df['text'][h].splitlines()[0][:30]}")

同じクラスどうしを結ぶ辺の割合: 互いに近傍の辺 0.697 / 一方向の辺 0.530 / 次数の中央値 36
次数 218 / クラス dokujo-tsushin / 同じクラスの割合: 自身の k 近傍 0.87, 一方向の近傍 188 件 0.75 / 別のクラスの近傍 51 件 | 30代男性がグッとくる口説き文句とは
次数 136 / クラス dokujo-tsushin / 同じクラスの割合: 自身の k 近傍 0.87, 一方向の近傍 106 件 0.75 / 別のクラスの近傍 30 件 | 2012年 独女たちの結婚観とは
次数 133 / クラス dokujo-tsushin / 同じクラスの割合: 自身の k 近傍 0.90, 一方向の近傍 103 件 0.81 / 別のクラスの近傍 23 件 | モテる男が選ぶ女子の条件
次数 115 / クラス smax / 同じクラスの割合: 自身の k 近傍 0.97, 一方向の近傍 85 件 0.73 / 別のクラスの近傍 24 件 | NTTドコモ、Xi対応Android 4.0 ICS搭載スマ
次数 113 / クラス dokujo-tsushin / 同じクラスの割合: 自身の k 近傍 0.97, 一方向の近傍 83 件 0.72 / 別のクラスの近傍 24 件 | 独身男性は独女より人妻と遊びたいってホント？


- グラフに基づく半教師あり学習は、「似ている文書は同じラベルを持つ」という仮定で、少数のラベルをグラフの辺に沿って広げる。ラベルが極端に少ないときに効果が大きい。
- 結果はグラフの作り方に強く左右される。普通の k 近傍グラフでは、多くの文書の近傍に入る「ハブ」ができる。この実験のハブは、近傍の大半が同じクラスだが、次数が大きいぶん、別のクラスの文書も数十件つながり、ラベルの通り道になる。ハブの辺の多くは片方の k 近傍にだけ入る一方向の辺で、一方向の辺は、互いに近傍である辺より別のクラスどうしを結ぶ割合が高い。
- 相互 k 近傍グラフ（互いに近傍である組だけを結ぶ）はハブができず、同じクラスどうしを結ぶ辺の割合が高い。その代わり、どこにもつながらない孤立点ができる。この実験では、分類の正解率の差は小さく、ラベル数によって優劣が入れ替わった。
- ラベルが増えると、普通の教師あり学習との差は縮まり、1 クラス 20 件では逆転している。数十件以上のラベルがあるなら、まずは事前学習済みモデルのファインチューニングや、埋め込み + 分類器（テキスト分類の章の「文埋め込みを特徴量にして少ない学習データで分類する」）を試す。
- 類似度に文埋め込みを使えば、同じ方法をそのまま適用できる。高次元の埋め込みでもハブは生じる（ハブネス問題）。
- 少数の例から語の集合を広げるブートストラッピングでも、ハブ（一般的な語）を通って無関係な語に広がる「意味ドリフト」が起きる（語の意味と意味の変化の章の「ブートストラッピングの意味ドリフトを観察して抑える」）。
- 実務でライブラリを使って同じことをする版は、テキスト分類の章の「少数のラベルをグラフで広げて分類する（scikit-learn の LabelSpreading）」。